# Experiment 01 — Data Validation

This notebook performs a comprehensive data quality check on the Austrian
electricity load dataset before any modelling begins.

**Steps:**
1. Locate and load the dataset
2. Inspect shape, columns, dtypes
3. Check timestamp range and frequency
4. Detect duplicates and missing values
5. Display chronological train / validation / test boundaries

In [ ]:
# ── Cell 1: Environment Setup ────────────────────────────────────
from pathlib import Path
import subprocess, sys

PROJECT_ROOT = Path("/kaggle/working/stlf-entso-2026")

if not PROJECT_ROOT.exists():
    subprocess.run(
        ["git", "clone",
         "https://github.com/AlvinHarist/stlf-entso-2026.git",
         str(PROJECT_ROOT)],
        check=True,
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"sys.path includes project: {str(PROJECT_ROOT) in sys.path}")

In [ ]:
# ── Cell 2: Package Versions ─────────────────────────────────────
import platform, numpy, pandas, sklearn
print(f"Python        : {platform.python_version()}")
print(f"NumPy         : {numpy.__version__}")
print(f"Pandas        : {pandas.__version__}")
print(f"Scikit-learn  : {sklearn.__version__}")
try:
    import tensorflow as tf
    print(f"TensorFlow    : {tf.__version__}")
except ImportError:
    print("TensorFlow    : not installed (not needed for this notebook)")

In [ ]:
# ── Cell 3: Configuration ────────────────────────────────────────
import yaml

CONFIG_PATH = PROJECT_ROOT / "configs" / "baseline.yaml"
with open(CONFIG_PATH) as f:
    config = yaml.safe_load(f)

# --- Kaggle data path override ---
KAGGLE_DATA_DIR = Path("/kaggle/input/stlf-entso-2026")

# Auto-search for the CSV
DATA_PATH = None
if KAGGLE_DATA_DIR.exists():
    for p in KAGGLE_DATA_DIR.rglob("*.csv"):
        if "combined_AT" in p.name:
            DATA_PATH = p
            break

# Fallback: try project-local path
if DATA_PATH is None:
    fallback = PROJECT_ROOT / config["data"]["path"]
    if fallback.exists():
        DATA_PATH = fallback

# Fallback: try repo root
if DATA_PATH is None:
    fallback = PROJECT_ROOT / "df_combined_AT.csv"
    if fallback.exists():
        DATA_PATH = fallback

if DATA_PATH is None:
    raise FileNotFoundError(
        "Could not locate df_combined_AT.csv. "
        "Please check Kaggle Dataset mount or config path."
    )

print(f"Dataset path: {DATA_PATH}")
print(f"Config loaded from: {CONFIG_PATH}")

In [ ]:
# ── Cell 4: Load Dataset ─────────────────────────────────────────
from src.data.load_data import load_dataset, validate_hourly_index, print_data_summary

df = load_dataset(DATA_PATH, timestamp_col=config["data"]["timestamp_col"])
print_data_summary(df)

In [ ]:
# ── Cell 5: Validate Hourly Frequency ────────────────────────────
report = validate_hourly_index(df)

print("Hourly Index Validation Report")
print("=" * 40)
for k, v in report.items():
    if k != "missing_timestamps":
        print(f"  {k:20s}: {v}")

if report["n_missing"] > 0:
    print(f"\n  First missing timestamps (up to 50):")
    for ts in report["missing_timestamps"]:
        print(f"    {ts}")

if report["is_valid"]:
    print("\n✓ Hourly index is VALID — no gaps, no duplicates.")
else:
    print("\n✗ Hourly index has ISSUES — see above.")

In [ ]:
# ── Cell 6: Check Target & Features ──────────────────────────────
target_col = config["data"]["target_col"]
weather_features = config["data"]["weather_features"]

print("Target column check:")
if target_col in df.columns:
    print(f"  ✓ '{target_col}' found")
    print(f"    Range: {df[target_col].min():.1f} — {df[target_col].max():.1f} MW")
    print(f"    Mean:  {df[target_col].mean():.1f} MW")
    print(f"    NAs:   {df[target_col].isna().sum()}")
else:
    print(f"  ✗ '{target_col}' NOT FOUND")

print("\nWeather feature check:")
for feat in weather_features:
    if feat in df.columns:
        print(f"  ✓ '{feat}' — range [{df[feat].min():.2f}, {df[feat].max():.2f}], NAs={df[feat].isna().sum()}")
    else:
        print(f"  ✗ '{feat}' NOT FOUND")

In [ ]:
# ── Cell 7: Chronological Split Boundaries ───────────────────────
from src.data.preprocessing import chronological_split

train_df, val_df, test_df = chronological_split(
    df,
    train_ratio=config["split"]["train_ratio"],
    val_ratio=config["split"]["val_ratio"],
)

print("\nSplit Boundaries:")
print(f"  TRAIN : {train_df.index.min()} → {train_df.index.max()}  ({len(train_df)} rows)")
print(f"  VAL   : {val_df.index.min()} → {val_df.index.max()}  ({len(val_df)} rows)")
print(f"  TEST  : {test_df.index.min()} → {test_df.index.max()}  ({len(test_df)} rows)")

In [ ]:
# ── Cell 8: Visualise Target Over Time ───────────────────────────
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(train_df.index, train_df[target_col], label="Train", linewidth=0.4)
ax.plot(val_df.index, val_df[target_col], label="Validation", linewidth=0.4)
ax.plot(test_df.index, test_df[target_col], label="Test", linewidth=0.4)

ax.axvline(val_df.index.min(), color="grey", linestyle="--", alpha=0.7, label="Split")
ax.axvline(test_df.index.min(), color="grey", linestyle="--", alpha=0.7)

ax.set_title("Austrian Electricity Load — Chronological Split")
ax.set_ylabel("Load (MW)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()

RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
(RESULTS_DIR / "deterministic").mkdir(parents=True, exist_ok=True)
(RESULTS_DIR / "probabilistic").mkdir(parents=True, exist_ok=True)
fig.savefig(RESULTS_DIR / "data_split_overview.png", dpi=150)
plt.show()
print(f"Plot saved to {RESULTS_DIR / 'data_split_overview.png'}")

In [ ]:
# ── Cell 9: Summary ──────────────────────────────────────────────
print("=" * 60)
print("DATA VALIDATION COMPLETE")
print("=" * 60)
print(f"Dataset       : {DATA_PATH.name}")
print(f"Rows          : {len(df)}")
print(f"Columns       : {len(df.columns)}")
print(f"Timestamp     : {df.index.min()} → {df.index.max()}")
print(f"Frequency     : {'Hourly ✓' if report['is_valid'] else 'ISSUES ✗'}")
print(f"Missing vals  : {df.isna().sum().sum()}")
print(f"Target col    : {target_col}")
print(f"Weather cols  : {weather_features}")
print(f"Train         : {len(train_df)} rows ({len(train_df)/len(df)*100:.1f}%)")
print(f"Validation    : {len(val_df)} rows ({len(val_df)/len(df)*100:.1f}%)")
print(f"Test          : {len(test_df)} rows ({len(test_df)/len(df)*100:.1f}%)")
print("=" * 60)